#PySpark Practice Notebook

In [3]:
#Source Data
#Create a csv file for the following data
"""
product_id,product,country,sales
1,Product A,USA,100
2,Product B,USA,80
3,Product C,USA,30
1,Product A,Canada,60
2,Product B,Canada,40
4,Product D,UK,50
5,Product E,UK,20
1,Product A,Germany,70
3,Product C,Germany,90
4,Product D,Germany,40


"""

'\nproduct_id,product,country,sales\n1,Product A,USA,100\n2,Product B,USA,80\n3,Product C,USA,30\n1,Product A,Canada,60\n2,Product B,Canada,40\n4,Product D,UK,50\n5,Product E,UK,20\n1,Product A,Germany,70\n3,Product C,Germany,90\n4,Product D,Germany,40\n\n\n'

In [4]:
#Import Pyspark & other necessary functions
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [5]:
#Create SparkSession for app "Sales Data Analysis"
spark=SparkSession.builder.appName("Sales Data Analysis").getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/03 04:45:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/03 04:45:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [6]:
#Create a pyspark DataFrame from the csv file on local storage
#Use StructType Declaration for column creation
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("country", StringType(), True),
    StructField("sales", IntegerType(), True)
])



In [7]:
#Verify schema for the newly created file
df = spark.read.csv("data.csv",header=True,schema=schema)
df.show()

+----------+---------+-------+-----+
|product_id|  product|country|sales|
+----------+---------+-------+-----+
|         1|Product A|    USA|  100|
|         2|Product B|    USA|   80|
|         3|Product C|    USA|   30|
|         1|Product A| Canada|   60|
|         2|Product B| Canada|   40|
|         4|Product D|     UK|   50|
|         5|Product E|     UK|   20|
|         1|Product A|Germany|   70|
|         3|Product C|Germany|   90|
|         4|Product D|Germany|   40|
+----------+---------+-------+-----+



In [8]:
#Display all entries for country "Germany"
df.filter("country='Germany'").show()

+----------+---------+-------+-----+
|product_id|  product|country|sales|
+----------+---------+-------+-----+
|         1|Product A|Germany|   70|
|         3|Product C|Germany|   90|
|         4|Product D|Germany|   40|
+----------+---------+-------+-----+



In [9]:
#Find the total number of products
df.groupby("product").count().show()

+---------+-----+
|  product|count|
+---------+-----+
|Product A|    3|
|Product B|    2|
|Product C|    2|
|Product D|    2|
|Product E|    1|
+---------+-----+



In [10]:
#Find top 3 products
df.groupby("product").agg(F.sum("Sales").alias("Sum_sales")).orderBy(F.desc("Sum_sales")).show(3)

+---------+---------+
|  product|Sum_sales|
+---------+---------+
|Product A|      230|
|Product B|      120|
|Product C|      120|
+---------+---------+
only showing top 3 rows


In [11]:
#Calculate total sales
df1=df.select(F.sum("sales").alias("Total_sales")).collect()[0][0]
print(df1)

580


In [12]:
#Find the market share of all products individually
df.groupby("product").agg((F.sum("sales")*100/df1).alias("Market Share")).show()

+---------+------------------+
|  product|      Market Share|
+---------+------------------+
|Product A|  39.6551724137931|
|Product B|20.689655172413794|
|Product C|20.689655172413794|
|Product D|15.517241379310345|
|Product E|3.4482758620689653|
+---------+------------------+

